# Oil Data / Solver Validation

Sanity-check pass on the crude oil dataset (`data/processed/cleaned_edges.csv`) now that the pipeline has switched commodities from gallium to oil (HS 2709) and moved to severity-based shocks.

1. Load `SupplyChainNetwork` on the oil data and confirm the graph builds and `simulate_shock` produces sensible results.
2. Run `find_rerouting_options` (the greedy baseline) and `run_comparison` (greedy vs. `min_cost_flow` vs. `or_tools`) on the same scenario and compare outputs.

Scenario used throughout: a Saudi Arabia export shock. Saudi Arabia is the top oil exporter by value in this dataset, so it's a realistic, high-impact case to sanity-check against.


## Imports

In [1]:
import sys
from pathlib import Path

project_root = Path("..")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd

from src.supply_chain_network import SupplyChainNetwork
from src.solvers.compare import run_comparison, greedy_solver
from src.solvers import min_cost_flow, or_tools

pd.set_option("display.max_columns", None)

In [2]:
network = SupplyChainNetwork(
    edges_path="../data/processed/cleaned_edges.csv",
    nodes_path="../data/processed/cleaned_nodes.csv",
    centroids_path="../data/processed/country_centroids.csv"
)

print(f"Nodes: {network.baseline['n_nodes']}, Edges: {network.baseline['n_edges']}")
print(f"Total trade value: ${network.baseline['total_trade_value_usd']:,.0f}")

Nodes: 153, Edges: 1040
Total trade value: $1,328,534,389,040


## Simulate shock on oil network

A shock of 60% reduction to Saudi Arabia's oil output results in roughly 9% of the total trade value of the network being displaced. 

Fully losing the Saudia Arabia node (a 100% shock to their oil output) results in 14% of total trade value being displaced.

In [3]:
TARGET_COUNTRY = "Saudi Arabia"
SEVERITY = 0.6                  # simulates a disruption to 60% of Saudi Arabia's oil output

shock_result = network.simulate_shock({TARGET_COUNTRY: SEVERITY})
assert shock_result["success"], shock_result.get("error")

shock_result

{'success': True,
 'error': None,
 'scenario': {'shocks': [{'country': 'Saudi Arabia', 'severity': 0.6}]},
 'baseline': {'n_edges': 1040,
  'n_nodes': 153,
  'n_components': 1,
  'largest_component_size': 153,
  'total_trade_value_usd': 1328534389039.58},
 'after_removal': {'n_edges': 1040,
  'n_nodes': 153,
  'n_components': 1,
  'largest_component_size': 153,
  'total_trade_value_usd': 1215838094748.82},
 'impact': {'trade_value_lost_usd': 112696294290.76,
  'pct_trade_value_lost': 8.48,
  'components_before': 1,
  'components_after': 1,
  'largest_component_before': 153,
  'largest_component_after': 153,
  'network_fragmented': False,
  'newly_isolated_countries': []},
 'centrality_shifts': {'description': 'Countries gaining structural importance (betweenness centrality) after the shock, i.e. where risk cascades to.',
  'top_gainers': []}}

In [4]:
full_shock = network.simulate_shock({TARGET_COUNTRY: 1.0})      # simulate fully losing the node

full_shock

{'success': True,
 'error': None,
 'scenario': {'shocks': [{'country': 'Saudi Arabia', 'severity': 1.0}]},
 'baseline': {'n_edges': 1040,
  'n_nodes': 153,
  'n_components': 1,
  'largest_component_size': 153,
  'total_trade_value_usd': 1328534389039.58},
 'after_removal': {'n_edges': 1011,
  'n_nodes': 153,
  'n_components': 2,
  'largest_component_size': 152,
  'total_trade_value_usd': 1140707231888.31},
 'impact': {'trade_value_lost_usd': 187827157151.27,
  'pct_trade_value_lost': 14.14,
  'components_before': 1,
  'components_after': 2,
  'largest_component_before': 153,
  'largest_component_after': 152,
  'network_fragmented': True,
  'newly_isolated_countries': ['Saudi Arabia']},
 'centrality_shifts': {'description': 'Countries gaining structural importance (betweenness centrality) after the shock, i.e. where risk cascades to.',
  'top_gainers': []}}

## Value weighted vulnerability scan

Top oil exporters ranked by structural fragmentation and economic disruption if removed.

Russia and the UAE being removed creates the most disruption to the structure of the graph, seperating it into 3 distinct components.

Saudia Arabia being removed from the graph leads to the largest economic disruption at 14% of trade value lost.

In [5]:
vulnerability_df = pd.DataFrame(network.rank_vulnerability(top_n=15))
vulnerability_df

,country,components_after,pct_trade_value_lost,n_isolated
0,Russian Federation,3,9.36,2
1,United Arab Emirates,3,8.63,2
2,Saudi Arabia,2,14.14,1
3,Iraq,2,7.39,1
4,Nigeria,2,3.08,1
5,Kazakhstan,2,2.88,1
6,Kuwait,2,2.47,1
7,Libya,2,2.10,1
8,Mexico,2,1.68,1
9,Venezuela,2,0.74,1


## Solver Comparison

Run the Saudi Arabia scenario through find_rerouting_options (greedy) directly, then run it through run_comparison to view greedy, min_cost_flow and or_tools results side by side.

In [6]:
reroute_result = network.find_rerouting_options({TARGET_COUNTRY: SEVERITY})
assert reroute_result["success"], reroute_result.get("error")

reroute_result["summary"]

{'n_displaced_relationships': 29,
 'total_displaced_value_usd': 112696294290.76,
 'total_unmet_value_usd': 0.0,
 'pct_covered': 100.0,
 'new_trade_relationships_formed': ['Brazil -> Iraq',
  'Brunei Darussalam -> Iraq',
  'Canada -> Iraq',
  'China -> Latvia',
  'Croatia -> Iraq',
  'India -> Central African Rep.',
  'India -> Italy',
  'India -> Marshall Isds',
  'India -> Poland',
  'Indonesia -> Iraq',
  'Lithuania -> Malaysia',
  'Netherlands -> Malta',
  'Pakistan -> Malaysia',
  'Poland -> Colombia',
  'Poland -> Cyprus',
  'Poland -> Ecuador',
  'Poland -> Hungary',
  'Poland -> Singapore',
  'Poland -> Sweden',
  'Poland -> Trinidad and Tobago',
  'Rep. of Korea -> Albania',
  'Rep. of Korea -> Finland',
  'Rep. of Korea -> Guatemala',
  'Rep. of Korea -> Ireland',
  'Rep. of Korea -> Rep. of Moldova',
  'South Africa -> Iraq',
  'Spain -> Malaysia',
  'Taiwan -> Barbados',
  'Taiwan -> Germany',
  'Taiwan -> Malaysia',
  'Taiwan -> Niger',
  'Taiwan -> Other Europe, nes',
  'T

In [7]:
greedy_allocations = pd.DataFrame([
    {"importer": r["importer"], "removed_supplier": r["removed_supplier"], **alloc}
    for r in reroute_result["reroutes"]
    for alloc in r["allocations"]
])
greedy_allocations.sort_values("allocated_value_usd", ascending=False).head(15)

,importer,removed_supplier,new_supplier,allocated_value_usd,landed_unit_cost_usd_per_kg,tariff_pct,tariff_methodology,is_new_trade_relationship,distance_km,est_supplier_lead_time_days
1,China,Saudi Arabia,United Arab Emirates,2.874604e+10,0.3824,0.04,existing_trade_relationship,False,5000.9,17.0
9,Japan,Saudi Arabia,Canada,1.724841e+10,0.4527,0.02,existing_trade_relationship,False,8082.7,23.2
8,Rep. of Korea,Saudi Arabia,Canada,1.199347e+10,0.4527,0.02,existing_trade_relationship,False,8578.9,24.2
2,Rep. of Korea,Saudi Arabia,United Arab Emirates,5.632644e+09,0.3751,0.02,existing_trade_relationship,False,7111.9,21.2
10,India,Saudi Arabia,Canada,4.468018e+09,0.4705,0.06,existing_trade_relationship,False,11469.1,29.9
16,India,Saudi Arabia,Mexico,4.182317e+09,0.5206,0.06,existing_trade_relationship,False,15094.4,37.2
34,Taiwan,Saudi Arabia,Malaysia,3.966251e+09,0.5727,0.02,importer_default_high_income,True,2973.6,57.9
35,Malaysia,Saudi Arabia,Iraq,3.378496e+09,0.5868,0.04,existing_trade_relationship,False,6827.6,20.7
26,Poland,Saudi Arabia,Ecuador,3.037872e+09,0.5401,0.02,importer_default_high_income,True,10670.0,73.3
11,India,Saudi Arabia,Venezuela,2.945399e+09,0.4801,0.06,existing_trade_relationship,False,15200.1,37.4


In [9]:
solvers = {
    "greedy": greedy_solver(network),
    "min_cost_flow": min_cost_flow.solve,
    "or_tools": or_tools.solve
}

comparison_df = run_comparison(network, {TARGET_COUNTRY: SEVERITY}, solvers)
comparison_df

,solver,success,objective_value,total_unmet_value_usd,pct_covered,n_new_relationships,runtime_seconds,error
0,greedy,True,5.316802e+10,0.0,100.0,35,0.0195,None
1,min_cost_flow,True,5.304323e+10,0.0,100.0,38,0.1027,None
2,or_tools,True,5.304336e+10,0.0,100.0,39,0.0434,None
